In [62]:
import torch
import torch.nn as nn
from torch.utils.data import random_split
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

In [36]:
dataset_path = "dataset/pomegranate_diseases_dataset"
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])


dataset = datasets.ImageFolder(
    root=dataset_path,
    transform=transform
)

In [37]:
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size


train_dataset, test_dataset = random_split(
    dataset,
    [train_size, test_size]
)

In [38]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

In [39]:
model = nn.Sequential(
    nn.Conv2d(
        in_channels=3,
        out_channels=16,
        kernel_size=3,
        padding=1
    ),
    nn.ReLU(),
    nn.MaxPool2d(2),

    nn.Conv2d(
        in_channels=16,
        out_channels=32,
        kernel_size=3,
        padding=1
    ),
    nn.ReLU(),
    nn.MaxPool2d(2),

    nn.Flatten(),
    nn.Linear(32*56*56,100),
    nn.ReLU(),

    nn.Linear(100,32),
    nn.ReLU(),

    nn.Linear(32,5)
)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

criterion = nn.CrossEntropyLoss()

In [40]:
def train(x,y):
    output = model(x)
    loss = criterion(output,y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    return loss.item()

In [ ]:
epoch = 10
for k in range(epoch):
    batches = 0
    imgs = 0
    total_loss = 0
    for images,labels in train_loader:
        x = images
        y = labels
        total_loss = total_loss + train(x,y)
        batches = batches + 1
        imgs = imgs + 32
        print("\rImages:",imgs,end="")
    
    print(f"\nEpoch : {k+1} || Loss : {total_loss/(batches)}")

torch.save(model.state_dict(), "agrox_model.pth")

Images: 4096
Epoch : 1 || Loss : 0.8160357628948987
Images: 4096
Epoch : 2 || Loss : 0.36256328847957775
Images: 4096
Epoch : 3 || Loss : 0.19253980959183536
Images: 4096
Epoch : 4 || Loss : 0.09973615549824899
Images: 4096
Epoch : 5 || Loss : 0.04133580531015468
Images: 4096
Epoch : 6 || Loss : 0.015742234875688155
Images: 4096
Epoch : 7 || Loss : 0.003318731252647922
Images: 4096
Epoch : 8 || Loss : 0.0008986363188228097
Images: 4096
Epoch : 9 || Loss : 0.0005336192399170159
Images: 4096
Epoch : 10 || Loss : 0.00038881444078242566


In [43]:
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for images, labels in test_loader:
        output = model(images)

        predictions = torch.argmax(output, dim=1)

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

accuracy = correct / total * 100

print(f"Test Accuracy: {accuracy:.2f} %")

Test Accuracy: 96.18 %


In [44]:
model.eval()

all_labels = []
all_predictions = []

with torch.no_grad():
    for images, labels in test_loader:
        output = model(images)
        predictions = torch.argmax(output, dim=1)

        all_labels.extend(labels.tolist())
        all_predictions.extend(predictions.tolist())

In [63]:
print(
    classification_report(
        all_labels,
        all_predictions,
        target_names=dataset.classes
    )
)

                  precision    recall  f1-score   support

      Alternaria       0.95      0.95      0.95       179
     Anthracnose       0.96      0.96      0.96       242
Bacterial_Blight       0.94      0.96      0.95       183
      Cercospora       0.96      0.91      0.93       117
         Healthy       0.99      0.99      0.99       299

        accuracy                           0.96      1020
       macro avg       0.96      0.95      0.96      1020
    weighted avg       0.96      0.96      0.96      1020



In [64]:
cm = confusion_matrix(all_labels, all_predictions)
print(cm)

[[170   1   3   3   2]
 [  1 233   6   2   0]
 [  4   3 175   0   1]
 [  2   6   2 107   0]
 [  2   0   1   0 296]]


In [61]:
total_params = sum(
    p.numel()
    for p in model.parameters()
)

print(f"Total parameters : {total_params/10**6 :.2f} Millions" )

Total parameters : 10.04 Millions
